<a href="https://colab.research.google.com/github/SOUMYABAN123/ANN-Project3/blob/main/ALLCNNBasedModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q kagglehub

In [2]:
import sys
import site
import subprocess

user_site = site.getusersitepackages()
if user_site and user_site not in sys.path:
    sys.path.append(user_site)

try:
    import kagglehub
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "kagglehub"])
    if user_site and user_site not in sys.path:
        sys.path.append(user_site)
    import kagglehub

# Download dataset
path = kagglehub.dataset_download("mostafaabla/garbage-classification")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'garbage-classification' dataset.
Path to dataset files: /kaggle/input/garbage-classification


In [3]:
from pathlib import Path

root_dir = Path(path)
print("Dataset path:", root_dir)

if not root_dir.exists():
    raise FileNotFoundError("Dataset path does not exist. Re-run the download cell.")

first_level_dirs = sorted([p for p in root_dir.iterdir() if p.is_dir()])

data_dir = root_dir
if len(first_level_dirs) == 1:
    possible_inner = first_level_dirs[0]
    inner_dirs = sorted([p for p in possible_inner.iterdir() if p.is_dir()])
    if len(inner_dirs) > 0:
        data_dir = possible_inner

print("Using data folder:", data_dir)

class_dirs = sorted([p for p in data_dir.iterdir() if p.is_dir()])
print(f"\nFound {len(class_dirs)} class folders:")
print([p.name for p in class_dirs])

for class_dir in class_dirs:
    image_files = [p for p in class_dir.iterdir() if p.is_file()]
    print(f"{class_dir.name}: {len(image_files)} images")

Dataset path: /kaggle/input/garbage-classification
Using data folder: /kaggle/input/garbage-classification/garbage_classification

Found 12 class folders:
['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']
battery: 945 images
biological: 985 images
brown-glass: 607 images
cardboard: 891 images
clothes: 5325 images
green-glass: 629 images
metal: 769 images
paper: 1050 images
plastic: 865 images
shoes: 1977 images
trash: 697 images
white-glass: 775 images


In [4]:
import os
import copy
import time
import random
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

In [5]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
base_dataset = datasets.ImageFolder(root=data_dir)

class_names = base_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Total images before balancing:", len(base_dataset))

Classes: ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']
Number of classes: 12
Total images before balancing: 15515


In [7]:
TARGET_PER_CLASS = 607

samples = base_dataset.samples
filepaths = [s[0] for s in samples]
labels = [s[1] for s in samples]

class_to_files = defaultdict(list)
for fp, label in zip(filepaths, labels):
    class_to_files[label].append(fp)

balanced_filepaths = []
balanced_labels = []

for label in sorted(class_to_files.keys()):
    class_files = class_to_files[label]
    random.shuffle(class_files)

    if len(class_files) < TARGET_PER_CLASS:
        raise ValueError(
            f"Class '{class_names[label]}' has only {len(class_files)} images, "
            f"which is less than {TARGET_PER_CLASS}."
        )

    selected_files = class_files[:TARGET_PER_CLASS]
    balanced_filepaths.extend(selected_files)
    balanced_labels.extend([label] * TARGET_PER_CLASS)

combined = list(zip(balanced_filepaths, balanced_labels))
random.shuffle(combined)
balanced_filepaths, balanced_labels = zip(*combined)

balanced_filepaths = list(balanced_filepaths)
balanced_labels = list(balanced_labels)

print("Total images after balancing:", len(balanced_filepaths))
print("\nBalanced class counts:")
for i, cls in enumerate(class_names):
    print(f"{cls}: {sum(1 for y in balanced_labels if y == i)}")

Total images after balancing: 7284

Balanced class counts:
battery: 607
biological: 607
brown-glass: 607
cardboard: 607
clothes: 607
green-glass: 607
metal: 607
paper: 607
plastic: 607
shoes: 607
trash: 607
white-glass: 607


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    balanced_filepaths,
    balanced_labels,
    test_size=0.30,
    stratify=balanced_labels,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)

print("Train:", len(X_train))
print("Val  :", len(X_val))
print("Test :", len(X_test))

print("\nClass distribution after split:")
for i, cls in enumerate(class_names):
    print(
        f"{cls} -> "
        f"Train: {sum(1 for y in y_train if y == i)}, "
        f"Val: {sum(1 for y in y_val if y == i)}, "
        f"Test: {sum(1 for y in y_test if y == i)}"
    )

Train: 5098
Val  : 1093
Test : 1093

Class distribution after split:
battery -> Train: 425, Val: 91, Test: 91
biological -> Train: 425, Val: 91, Test: 91
brown-glass -> Train: 425, Val: 91, Test: 91
cardboard -> Train: 424, Val: 91, Test: 92
clothes -> Train: 425, Val: 91, Test: 91
green-glass -> Train: 425, Val: 91, Test: 91
metal -> Train: 424, Val: 92, Test: 91
paper -> Train: 425, Val: 91, Test: 91
plastic -> Train: 425, Val: 91, Test: 91
shoes -> Train: 425, Val: 91, Test: 91
trash -> Train: 425, Val: 91, Test: 91
white-glass -> Train: 425, Val: 91, Test: 91


In [9]:
class CustomImageDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [10]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [11]:
train_dataset = CustomImageDataset(X_train, y_train, transform=train_transform)
val_dataset   = CustomImageDataset(X_val, y_val, transform=test_transform)
test_dataset  = CustomImageDataset(X_test, y_test, transform=test_transform)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("DataLoaders ready.")

DataLoaders ready.


In [12]:
def get_model(model_name, num_classes):
    model_name = model_name.lower()

    if model_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "vgg16":
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return model

In [13]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    preds_all = []
    labels_all = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(labels_all, preds_all)
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds_all = []
    labels_all = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(labels_all, preds_all)
    return epoch_loss, epoch_acc, labels_all, preds_all

In [14]:
def train_and_evaluate_model(model_name, train_loader, val_loader, test_loader, class_names, device, num_epochs=20):
    print(f"\n{'='*70}")
    print(f"Training model: {model_name}")
    print(f"{'='*70}")

    model = get_model(model_name, len(class_names))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    start_time = time.time()

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        scheduler.step()

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print("Best model updated.")

        print("-" * 50)

    elapsed = time.time() - start_time
    print(f"Training complete in {elapsed/60:.2f} minutes")
    print(f"Best validation accuracy: {best_val_acc:.4f}")

    model.load_state_dict(best_model_wts)

    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Acc : {test_acc:.4f}")

    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    overall_results = pd.DataFrame([
        {
            "Models": model_name,
            "Accuracy": round(test_acc, 4),
            "Precision": round(precision_w, 4),
            "Recall": round(recall_w, 4),
            "F1 Score": round(f1_w, 4),
            "Averaging Approach": "Weighted"
        },
        {
            "Models": model_name,
            "Accuracy": round(test_acc, 4),
            "Precision": round(precision_m, 4),
            "Recall": round(recall_m, 4),
            "F1 Score": round(f1_m, 4),
            "Averaging Approach": "Macro"
        }
    ])

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    classwise_rows = []
    for cls in class_names:
        classwise_rows.append({
            "Model": model_name,
            "Class Name": cls,
            "Accuracy": round(test_acc, 4),
            "Precision": round(report_dict[cls]["precision"], 4),
            "Recall": round(report_dict[cls]["recall"], 4),
            "F1 Score": round(report_dict[cls]["f1-score"], 4),
            "Support": int(report_dict[cls]["support"])
        })

    classwise_results = pd.DataFrame(classwise_rows)

    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

    history_df = pd.DataFrame(history)

    return model, overall_results, classwise_results, cm_df, history_df

In [15]:
model_names = [
    "densenet121",
    "efficientnet_b0",
    "resnet18",
    "vgg16"
]

all_overall_results = []
all_classwise_results = []

trained_models = {}

for model_name in model_names:
    model, overall_df, classwise_df, cm_df, history_df = train_and_evaluate_model(
        model_name=model_name,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        class_names=class_names,
        device=device,
        num_epochs=20
    )

    trained_models[model_name] = model
    all_overall_results.append(overall_df)
    all_classwise_results.append(classwise_df)

    overall_csv = f"{model_name}_overall_results_balanced.csv"
    classwise_csv = f"{model_name}_classwise_results_balanced.csv"
    history_csv = f"{model_name}_training_history_balanced.csv"
    confusion_csv = f"{model_name}_confusion_matrix_balanced.csv"
    model_path = f"{model_name}_balanced_607_per_class.pth"

    overall_df.to_csv(overall_csv, index=False)
    classwise_df.to_csv(classwise_csv, index=False)
    history_df.to_csv(history_csv, index=False)
    cm_df.to_csv(confusion_csv, index=True)
    torch.save(model.state_dict(), model_path)

    print(f"Saved files for {model_name}:")
    print(overall_csv)
    print(classwise_csv)
    print(history_csv)
    print(confusion_csv)
    print(model_path)
    print("=" * 70)


Training model: densenet121
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 121MB/s]


Epoch [1/20]
Train Loss: 0.8144 | Train Acc: 0.7854
Val   Loss: 0.2510 | Val   Acc: 0.9323
Best model updated.
--------------------------------------------------
Epoch [2/20]
Train Loss: 0.2556 | Train Acc: 0.9274
Val   Loss: 0.1784 | Val   Acc: 0.9515
Best model updated.
--------------------------------------------------
Epoch [3/20]
Train Loss: 0.1438 | Train Acc: 0.9594
Val   Loss: 0.1427 | Val   Acc: 0.9597
Best model updated.
--------------------------------------------------
Epoch [4/20]
Train Loss: 0.0996 | Train Acc: 0.9727
Val   Loss: 0.1313 | Val   Acc: 0.9625
Best model updated.
--------------------------------------------------
Epoch [5/20]
Train Loss: 0.0782 | Train Acc: 0.9788
Val   Loss: 0.1416 | Val   Acc: 0.9570
--------------------------------------------------
Epoch [6/20]
Train Loss: 0.0413 | Train Acc: 0.9916
Val   Loss: 0.1076 | Val   Acc: 0.9726
Best model updated.
--------------------------------------------------
Epoch [7/20]
Train Loss: 0.0341 | Train Acc: 0.9

100%|██████████| 20.5M/20.5M [00:00<00:00, 179MB/s]


Epoch [1/20]
Train Loss: 1.3587 | Train Acc: 0.6514
Val   Loss: 0.4305 | Val   Acc: 0.8957
Best model updated.
--------------------------------------------------
Epoch [2/20]
Train Loss: 0.4062 | Train Acc: 0.8801
Val   Loss: 0.2204 | Val   Acc: 0.9460
Best model updated.
--------------------------------------------------
Epoch [3/20]
Train Loss: 0.2291 | Train Acc: 0.9325
Val   Loss: 0.1608 | Val   Acc: 0.9579
Best model updated.
--------------------------------------------------
Epoch [4/20]
Train Loss: 0.1603 | Train Acc: 0.9519
Val   Loss: 0.1360 | Val   Acc: 0.9616
Best model updated.
--------------------------------------------------
Epoch [5/20]
Train Loss: 0.1118 | Train Acc: 0.9725
Val   Loss: 0.1288 | Val   Acc: 0.9652
Best model updated.
--------------------------------------------------
Epoch [6/20]
Train Loss: 0.0885 | Train Acc: 0.9772
Val   Loss: 0.1206 | Val   Acc: 0.9671
Best model updated.
--------------------------------------------------
Epoch [7/20]
Train Loss: 0.0

100%|██████████| 44.7M/44.7M [00:00<00:00, 125MB/s]


Epoch [1/20]
Train Loss: 0.6745 | Train Acc: 0.8005
Val   Loss: 0.2936 | Val   Acc: 0.9131
Best model updated.
--------------------------------------------------
Epoch [2/20]
Train Loss: 0.2444 | Train Acc: 0.9257
Val   Loss: 0.2292 | Val   Acc: 0.9305
Best model updated.
--------------------------------------------------
Epoch [3/20]
Train Loss: 0.1540 | Train Acc: 0.9553
Val   Loss: 0.2124 | Val   Acc: 0.9323
Best model updated.
--------------------------------------------------
Epoch [4/20]
Train Loss: 0.1027 | Train Acc: 0.9729
Val   Loss: 0.1956 | Val   Acc: 0.9405
Best model updated.
--------------------------------------------------
Epoch [5/20]
Train Loss: 0.0700 | Train Acc: 0.9808
Val   Loss: 0.1931 | Val   Acc: 0.9424
Best model updated.
--------------------------------------------------
Epoch [6/20]
Train Loss: 0.0440 | Train Acc: 0.9912
Val   Loss: 0.1429 | Val   Acc: 0.9634
Best model updated.
--------------------------------------------------
Epoch [7/20]
Train Loss: 0.0

100%|██████████| 528M/528M [00:01<00:00, 286MB/s]


Epoch [1/20]
Train Loss: 0.8169 | Train Acc: 0.7395
Val   Loss: 0.3612 | Val   Acc: 0.8847
Best model updated.
--------------------------------------------------
Epoch [2/20]
Train Loss: 0.3646 | Train Acc: 0.8827
Val   Loss: 0.3103 | Val   Acc: 0.8957
Best model updated.
--------------------------------------------------
Epoch [3/20]
Train Loss: 0.2624 | Train Acc: 0.9162
Val   Loss: 0.3088 | Val   Acc: 0.9003
Best model updated.
--------------------------------------------------
Epoch [4/20]
Train Loss: 0.2063 | Train Acc: 0.9361
Val   Loss: 0.3020 | Val   Acc: 0.9003
--------------------------------------------------
Epoch [5/20]
Train Loss: 0.2000 | Train Acc: 0.9378
Val   Loss: 0.3491 | Val   Acc: 0.9021
Best model updated.
--------------------------------------------------
Epoch [6/20]
Train Loss: 0.0625 | Train Acc: 0.9829
Val   Loss: 0.2175 | Val   Acc: 0.9259
Best model updated.
--------------------------------------------------
Epoch [7/20]
Train Loss: 0.0366 | Train Acc: 0.9

In [16]:
final_overall_results = pd.concat(all_overall_results, ignore_index=True)
final_overall_results

,Models,Accuracy,Precision,Recall,F1 Score,Averaging Approach
0,densenet121,0.9625,0.9633,0.9625,0.9624,Weighted
1,densenet121,0.9625,0.9632,0.9625,0.9624,Macro
2,efficientnet_b0,0.9616,0.9621,0.9616,0.9613,Weighted
3,efficientnet_b0,0.9616,0.9620,0.9616,0.9613,Macro
4,resnet18,0.9625,0.9633,0.9625,0.9623,Weighted
5,resnet18,0.9625,0.9633,0.9625,0.9623,Macro
6,vgg16,0.9506,0.9512,0.9506,0.9505,Weighted
7,vgg16,0.9506,0.9512,0.9506,0.9504,Macro


In [17]:
final_classwise_results = pd.concat(all_classwise_results, ignore_index=True)
final_classwise_results

,Model,Class Name,Accuracy,Precision,Recall,F1 Score,Support
0,densenet121,battery,0.9625,0.9468,0.9780,0.9622,91
1,densenet121,biological,0.9625,0.9783,0.9890,0.9836,91
2,densenet121,brown-glass,0.9625,0.9888,0.9670,0.9778,91
3,densenet121,cardboard,0.9625,1.0000,0.9565,0.9778,92
4,densenet121,clothes,0.9625,0.9891,1.0000,0.9945,91
5,densenet121,green-glass,0.9625,0.9677,0.9890,0.9783,91
6,densenet121,metal,0.9625,0.8713,0.9670,0.9167,91
7,densenet121,paper,0.9625,0.9560,0.9560,0.9560,91
8,densenet121,plastic,0.9625,0.9524,0.8791,0.9143,91
9,densenet121,shoes,0.9625,0.9783,0.9890,0.9836,91


In [18]:
final_overall_results.to_csv("all_models_overall_results_balanced.csv", index=False)
final_classwise_results.to_csv("all_models_classwise_results_balanced.csv", index=False)

print("Saved combined result files:")
print("all_models_overall_results_balanced.csv")
print("all_models_classwise_results_balanced.csv")

Saved combined result files:
all_models_overall_results_balanced.csv
all_models_classwise_results_balanced.csv
